# BrickPilot learned planner: conditional ridge + retrieval
This notebook reads the project's source and saved model without training or modifying them. The production inference is **TypeScript**; Python cells below inspect its data and demonstrate its calculations.

Flow: ground-floor brief → 4 encoded features → standardized ridge prediction → 12 nearest exemplars → seeded weighted choice → 25% ridge + 75% exemplar → guidance for the rule-based planner.

“Conditional” means conditioned on the brief; this code fits ordinary multi-output ridge regression, not a separate special algorithm. It returns footprint fill and living/kitchen anchors, plus the retrieved family and aspect, rather than walls or a validated floor plan.

In [1]:
from pathlib import Path
import json, math
from collections import Counter
from IPython.display import display, Markdown, Code
ROOT = Path.cwd()
if not (ROOT / 'src/lib/engine/planner/ml/proposal.ts').exists():
    ROOT = Path(r'C:\Users\aryan\Documents\projects\brick-pilot-final')
proposal_path = ROOT / 'src/lib/engine/planner/ml/proposal.ts'
model_path = ROOT / 'src/lib/engine/planner/ml/data/plan-model.json'
training_path = ROOT / 'scripts/train-plan-model.py'
model = json.loads(model_path.read_text(encoding='utf-8'))
print('Project:', ROOT)


Project: C:\Users\aryan\Documents\projects\brick-pilot-final


## 1. Saved model metadata and evaluation
The metrics are for ridge predictions before runtime clamping and retrieval blending. They do not measure final floor-plan validity or the complete blended planner.

In [2]:
metadata = {k:v for k,v in model.items() if k not in ('weights','exemplars','mean','scale')}
print(json.dumps(metadata, indent=2))
print('Retrieved memory size:', len(model['exemplars']))
print('Families:', dict(Counter(e['family'] for e in model['exemplars'])))


{
  "version": "resplan-ridge-retrieval-v1",
  "method": "conditional-ridge + seeded-nearest-exemplar",
  "source": "ResPlan",
  "sourceUrl": "https://github.com/m-agour/ResPlan",
  "licence": "CC BY 4.0",
  "archiveSha256": "e8799e74e6d3d830c1d2fad5f6f953ee1aad43214a12c5d60c7cd7404ef81ade",
  "featureNames": [
    "bedrooms/8",
    "bathrooms/8",
    "enclosedAreaM2/600",
    "logAspect/log4"
  ],
  "targetNames": [
    "fill",
    "livingX",
    "livingY",
    "kitchenX",
    "kitchenY"
  ],
  "alpha": 100.0,
  "trainingCount": 7296,
  "metrics": {
    "validation": {
      "count": 910,
      "modelMSE": 0.03234025811786591,
      "meanBaselineMSE": 0.03306710660460672
    },
    "test": {
      "count": 921,
      "modelMSE": 0.03209185789298488,
      "meanBaselineMSE": 0.03314799943935506
    }
  }
}
Retrieved memory size: 192
Families: {'rectangular': 48, 'stepped': 48, 'l-shape': 48, 'courtyard': 48}


## 2. Production inference code
Feature encoding uses indoor ground-floor rooms. Bedroom detection follows the source role/id rule; bathroom detection follows its wet-room exclusions. Area is target enclosed area in m²; aspect is envelope width/depth. Model schema validation can disable learned guidance if loading fails.

In [3]:
display(Code(proposal_path.read_text(encoding='utf-8'), language='typescript'))

import { z } from 'zod'
import rawModel from './data/plan-model.json' with { type: 'json' }
import type { CanonicalModel } from '../../../model/canonical.ts'
import { makeRng } from '../../massing/rng.ts'

const finite=z.number().finite()
const family=z.enum(['rectangular','stepped','l-shape','courtyard'])
const modelSchema=z.object({
  version:z.literal('resplan-ridge-retrieval-v1'), mean:z.array(finite).length(4),
  scale:z.array(finite.positive()).length(4), weights:z.array(z.array(finite).length(5)).length(5),
  trainingCount:z.number().int().positive(),
  exemplars:z.array(z.object({id:z.number().int(),features:z.array(finite).length(4),
    targets:z.array(finite).length(5),aspect:finite.positive(),family,mask:z.array(z.union([z.literal(0),z.literal(1)])).length(144)})).min(1),
})
export const parsePlanModel=(input:unknown)=>modelSchema.safeParse(input)
const parsed=parsePlanModel(rawModel)
export const learnedPlanModel=parsed.success?parsed.data:null
export type PlanProposal={version:string;exemplarId:number;family:z.infer<typeof family>;
  aspect:number;fill:number;living:[number,number];kitchen:[number,number];distance:number}
const clamp=(n:number,lo=0,hi=1)=>Math.max(lo,Math.min(hi,n))
export const PLAN_LEARNING_LIMITS={neighbors:12,regressionWeight:.25,exemplarWeight:.75,
  aspectWeight:.6,livingWeight:.25,kitchenWeight:.15,candidatePool:8} as const
/** Same encoding and units as train-plan-model.py. Ground-floor room programme only. */
export function briefFeatures(model:CanonicalModel):number[]{
  const rooms=model.floors[0].spaces.filter(r=>!r.outdoor)
  const bedrooms=rooms.filter(r=>r.role!==undefined||r.id.startsWith('bed')).length
  const baths=rooms.filter(r=>r.wet&&r.id!=='kitchen'&&r.id!=='utility').length
  return [Math.min(bedrooms,8)/8,Math.min(baths,8)/8,
    Math.min(rooms.reduce((sum,r)=>sum+r.target,0),600)/600,
    Math.log(clamp(model.envelope.width/model.envelope.depth,.25,4))/Math.log(4)]
}
/** Learned conditional prediction plus seed-controlled sampling among near neighbours.
 * Returns guidance, never unvalidated room coordinates or wall/door geometry. */
export function proposePlan(model:CanonicalModel,seed:number):PlanProposal|null{
  const trained=learnedPlanModel
  if(!trained)return null
  const features=briefFeatures(model)
  const standardized=features.map((n,i)=>(n-trained.mean[i])/trained.scale[i])
  const predicted=trained.weights[0].map((bias,j)=>clamp(bias+standardized.reduce((sum,n,i)=>sum+n*trained.weights[i+1][j],0)))
  const neighbors=trained.exemplars.map(e=>({e,distance:e.features.reduce((sum,n,i)=>
    sum+((n-features[i])/trained.scale[i])**2,0)})).sort((a,b)=>a.distance-b.distance||a.e.id-b.e.id).slice(0,PLAN_LEARNING_LIMITS.neighbors)
  const rng=makeRng(seed,`${model.seed.split('-')[0]}|learned-plan-v1`)
  const chosen=rng.weighted(neighbors.map(n=>[n,1/(1+n.distance)] as const))
  const target=predicted.map((n,i)=>clamp(n*PLAN_LEARNING_LIMITS.regressionWeight+chosen.e.targets[i]*PLAN_LEARNING_LIMITS.exemplarWeight))
  return {version:trained.version,exemplarId:chosen.e.id,family:chosen.e.family,
    aspect:chosen.e.aspect,fill:target[0],living:[target[1],target[2]],
    kitchen:[target[3],target[4]],distance:chosen.distance}
}

## 3. Ridge coefficients
Rows are intercept, then four standardized input features. Columns are fill, living X/Y and kitchen X/Y. Training leaves the intercept unpenalized and chooses alpha from 0.1, 1, 10, 100 using validation MSE.

In [4]:
print('Feature names:', model['featureNames'])
print('Means:', model['mean'])
print('Scales:', model['scale'])
print('Target names:', model['targetNames'])
for name, row in zip(['intercept'] + model['featureNames'], model['weights']):
    print(f'{name:24}', ['%.6f' % x for x in row])


Feature names: ['bedrooms/8', 'bathrooms/8', 'enclosedAreaM2/600', 'logAspect/log4']
Means: [0.33009525767543857, 0.3158408717105263, 0.1838457270127624, 0.017969701871600356]
Scales: [0.09662351255913383, 0.09839537441160025, 0.1402743921059504, 0.30153684473570175]
Target names: ['fill', 'livingX', 'livingY', 'kitchenX', 'kitchenY']
intercept                ['0.779545', '0.496480', '0.631785', '0.490514', '0.516431']
bedrooms/8               ['0.007821', '-0.000770', '-0.005227', '-0.003329', '0.019725']
bathrooms/8              ['-0.018170', '0.002206', '-0.009819', '0.002129', '0.008783']
enclosedAreaM2/600       ['0.006903', '0.000138', '-0.002193', '0.000144', '-0.003068']
logAspect/log4           ['-0.000211', '-0.003103', '-0.034848', '0.004451', '-0.047314']


## 4. Inspect a sample brief
Change these four values and rerun. This demonstration reproduces feature encoding, ridge prediction and nearest-neighbor distances. The sample counts are supplied directly rather than derived from a CanonicalModel.

In [5]:
bedrooms, bathrooms, enclosed_area_m2 = 4, 3, 220
plot_width, plot_depth = 15, 20
clamp = lambda x: max(0, min(1, x))
features = [min(bedrooms,8)/8, min(bathrooms,8)/8,
            min(enclosed_area_m2,600)/600,
            math.log(max(.25,min(4,plot_width/plot_depth)))/math.log(4)]
z = [(x-m)/s for x,m,s in zip(features,model['mean'],model['scale'])]
predicted = [clamp(model['weights'][0][j] + sum(z[i]*model['weights'][i+1][j] for i in range(4))) for j in range(5)]
neighbors = sorted([(sum(((a-b)/s)**2 for a,b,s in zip(e['features'],features,model['scale'])),e) for e in model['exemplars']], key=lambda pair:(pair[0],pair[1]['id']))[:12]
print('Encoded features:', features)
print('Clamped ridge prediction:', dict(zip(model['targetNames'],predicted)))
total = sum(1/(1+d) for d,e in neighbors)
for d,e in neighbors:
    print('ID:',e['id'],'family:',e['family'],'distance:',round(d,4),'selection probability:',round((1/(1+d))/total,4))


Encoded features: [0.5, 0.375, 0.36666666666666664, -0.2075187496394219]
Clamped ridge prediction: {'fill': 0.7915280183221376, 'livingX': 0.49895184626216443, 'livingY': 0.6398898395333068, 'kitchenX': 0.48280011125313566, 'kitchenY': 0.5877791926506968}
ID: 12883 family: courtyard distance: 1.8026 selection probability: 0.1002
ID: 576 family: courtyard distance: 1.8212 selection probability: 0.0995
ID: 8734 family: rectangular distance: 1.8672 selection probability: 0.0979
ID: 2716 family: rectangular distance: 1.9069 selection probability: 0.0966
ID: 6347 family: courtyard distance: 2.5099 selection probability: 0.08
ID: 14801 family: stepped distance: 2.5973 selection probability: 0.078
ID: 9712 family: courtyard distance: 2.6683 selection probability: 0.0765
ID: 2841 family: courtyard distance: 2.7426 selection probability: 0.075
ID: 4078 family: rectangular distance: 2.763 selection probability: 0.0746
ID: 7731 family: rectangular distance: 2.7676 selection probability: 0.0745
ID

## 5. Inspect the blend and retrieved shape
For illustration, select one of the 12 neighbors by index. Production instead makes a seed-controlled weighted draw with probability proportional to 1/(1+distance); its exact random generator is displayed below. The 12×12 occupancy mask is stored training geometry; proposal.ts itself does not use it in its returned guidance. Family and aspect come directly from the chosen exemplar.

In [6]:
neighbor_index = 0
distance, chosen = neighbors[neighbor_index]
blended = [clamp(.25*p + .75*t) for p,t in zip(predicted,chosen['targets'])]
print('Illustrative choice:',chosen['id'],chosen['family'],'aspect:',chosen['aspect'])
print('Blended guidance:',dict(zip(model['targetNames'],blended)))
print('Exemplar metadata:',json.dumps({k:v for k,v in chosen.items() if k!='mask'},indent=2))
print('Occupancy (displayed with increasing Y upward):')
for y in reversed(range(12)):
    print(''.join('██' if value else '  ' for value in chosen['mask'][12*y:12*(y+1)]))


Illustrative choice: 12883 courtyard aspect: 0.6722983257229832
Blended guidance: {'fill': 0.736378861428587, 'livingX': 0.5046605246836013, 'livingY': 0.7379595858235057, 'kitchenX': 0.251689916497173, 'kitchenY': 0.6878360870709819}
Exemplar metadata: {
  "id": 12883,
  "features": [
    0.5,
    0.5,
    0.31802453614417703,
    -0.2864132690847695
  ],
  "targets": [
    0.7179958091307368,
    0.506563417490747,
    0.7706495012535721,
    0.17465318491185206,
    0.7211883852110769
  ],
  "aspect": 0.6722983257229832,
  "family": "courtyard"
}
Occupancy (displayed with increasing Y upward):
    ████████████████    
████████████████████    
████████████████████████
████████████████████████
████████████████████████
    ████████████████████
    ██████████████████  
      ████████████████  
        ██████████████  
        ██████████████  
            ██████████  
            ██████████  


In [7]:
display(Code((ROOT / 'src/lib/engine/massing/rng.ts').read_text(encoding='utf-8'), language='typescript'))

/* Deterministic PRNG for the massing grammar. A seed + the brief string fully
 * determine the house, so a run is bit-for-bit reproducible. */

/** FNV-1a → uint32 */
export function fnv(s: string): number {
  let h = 0x811c9dc5
  for (let i = 0; i < s.length; i++) {
    h ^= s.charCodeAt(i)
    h = Math.imul(h, 0x01000193)
  }
  return h >>> 0
}

export type Rng = {
  /** float in [0, 1) */
  next: () => number
  /** float in [lo, hi) */
  range: (lo: number, hi: number) => number
  /** integer in [lo, hi] */
  int: (lo: number, hi: number) => number
  /** true with probability p */
  chance: (p: number) => boolean
  /** uniform pick */
  pick: <T>(xs: readonly T[]) => T
  /** weighted pick — weights need not sum to 1 */
  weighted: <T>(entries: readonly [T, number][]) => T
  /** -1 or +1 */
  sign: () => 1 | -1
}

export function makeRng(seed: number, salt = ''): Rng {
  let a = (fnv(`${seed}|${salt}`) ^ 0x9e3779b9) >>> 0
  const next = () => {
    a |= 0
    a = (a + 0x6d2b79f5) | 0
    let t = Math.imul(a ^ (a >>> 15), 1 | a)
    t = (t + Math.imul(t ^ (t >>> 7), 61 | t)) ^ t
    return ((t ^ (t >>> 14)) >>> 0) / 4294967296
  }
  const range = (lo: number, hi: number) => lo + (hi - lo) * next()
  const int = (lo: number, hi: number) => Math.floor(lo + (hi - lo + 1) * next())
  const chance = (p: number) => next() < p
  const pick = <T>(xs: readonly T[]) => xs[Math.floor(next() * xs.length)]
  const weighted = <T>(entries: readonly [T, number][]): T => {
    const total = entries.reduce((s, [, w]) => s + Math.max(0, w), 0)
    let r = next() * total
    for (const [v, w] of entries) {
      r -= Math.max(0, w)
      if (r <= 0) return v
    }
    return entries[entries.length - 1][0]
  }
  const sign = () => (next() < 0.5 ? -1 : 1) as 1 | -1
  return { next, range, int, chance, pick, weighted, sign }
}

## 6. Training source
Training loads ResPlan through an allowlisted unpickler, excludes augmented plans, normalizes entrance orientation and geometry, deduplicates occupancy/anchors, and uses the supplied train/validation/test split. It builds a training-only memory of up to 48 examples per derived shape family. This cell displays the script; it does not execute training.

Limitations: single-floor source data; heuristic shape labels; deduplication does not prove absence of every near duplicate. At inference the envelope aspect is used as a feature, while training uses source footprint aspect.

In [8]:
display(Code(training_path.read_text(encoding='utf-8'), language='python'))

"""Train a small conditional footprint/anchor model from the user's ResPlan archive.
No dataset code is executed. Pickle globals are explicitly allowlisted.
Python dependencies are offline training tools only: numpy, shapely, networkx.
"""
import argparse
import hashlib
import io
import json
import pickle
import sys
import zipfile
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / 'output/ml-python'))
import numpy as np
import networkx as nx
from networkx.classes import reportviews, coreviews
from shapely import from_wkb
from shapely.ops import unary_union
from shapely import affinity


class RestrictedPlans(pickle.Unpickler):
    def find_class(self, module, name):
        allowed = {
            ('shapely.io', 'from_wkb'): from_wkb,
            ('numpy._core.multiarray', 'scalar'): np._core.multiarray.scalar,
            ('numpy.core.multiarray', 'scalar'): np._core.multiarray.scalar,
            ('numpy', 'dtype'): np.dtype,
            ('networkx.classes.graph', 'Graph'): nx.Graph,
            ('networkx.classes.coreviews', 'AdjacencyView'): coreviews.AdjacencyView,
            **{('networkx.classes.reportviews', n): getattr(reportviews, n)
               for n in ('NodeView', 'EdgeView', 'DegreeView')},
        }
        if (module, name) not in allowed:
            raise ValueError(f'Unsupported pickle class: {module}.{name}')
        return allowed[module, name]


def parts(g):
    if g is None or getattr(g, 'is_empty', True):
        return []
    return list(g.geoms) if hasattr(g, 'geoms') else [g]


def run():
    parser = argparse.ArgumentParser()
    parser.add_argument('archive')
    parser.add_argument('--inspect', action='store_true')
    args = parser.parse_args()
    with zipfile.ZipFile(args.archive) as archive:
        payload = archive.read('ResPlan.pkl')
        plans = RestrictedPlans(io.BytesIO(payload)).load()
        splits = json.loads(archive.read('split.json'))
        licence = archive.read('LICENSE').decode()
    if args.inspect:
        print(len(plans), list(plans[0]))
        for k,v in plans[0].items():
            print(k, str(v)[:100])
        return
    rows = []
    seen = set()
    # Deduplicate quantized, normalized shape + semantic anchors before splitting.
    # Canonical augmented plans are excluded entirely.
    aug = set(splits.get('augmented', []))
    train = set(splits['train'])
    val = set(splits['val'])
    test = set(splits['test'])
    dropped = 0
    for p in plans:
        if p['id'] in aug:
            continue
        rooms = {k:parts(p.get(k)) for k in ('bedroom','bathroom','kitchen','living','balcony','corridor')}
        polygons = [g for key,gs in rooms.items() if key!='balcony' for g in gs]
        polygons += parts(p.get('wall')) + parts(p.get('inner'))
        if not polygons:
            dropped += 1
            continue
        shape = unary_union([g if g.is_valid else g.buffer(0) for g in polygons])
        x,y,x1,y1 = shape.bounds
        w,h = x1-x,y1-y
        area = float(p.get('net_area',0))
        if w <= 0 or h <= 0 or not np.isfinite(area) or not 20 < area < 1000:
            dropped += 1
            continue
        # Some archive coordinates are image-scale. Learn normalized geometry;
        # area comes from recorded net_area, never from unverified polygon units.
        door = p.get('front_door')
        angle = 0
        if door is not None and not door.is_empty:
            c = door.centroid
            side = min([(abs(c.y-y)/h,180),(abs(c.y-y1)/h,0),
                        (abs(c.x-x)/w,-90),(abs(c.x-x1)/w,90)])[1]
            angle=side
        if angle:
            origin=shape.centroid
            shape=affinity.rotate(shape,angle,origin=origin)
            rooms={k:[affinity.rotate(g,angle,origin=origin) for g in gs] for k,gs in rooms.items()}
            x,y,x1,y1=shape.bounds;w,h=x1-x,y1-y
        def centre(key):
            if not rooms[key]: return [.5,.5]
            c = unary_union(r

## 7. Where the proposal is consumed
These source references help trace guidance into the planner. Open the listed files in the project for the surrounding implementation.

In [9]:
for path in sorted((ROOT/'src/lib/engine/planner').rglob('*.ts')):
    if path == proposal_path: continue
    for lineno,line in enumerate(path.read_text(encoding='utf-8').splitlines(),1):
        if 'proposePlan' in line or 'PLAN_LEARNING_LIMITS' in line:
            print(f'{path.relative_to(ROOT)}:{lineno}: {line.strip()}')


src\lib\engine\planner\index.ts:23: import { PLAN_LEARNING_LIMITS } from './ml/proposal.ts'
src\lib\engine\planner\index.ts:95: return PLAN_LEARNING_LIMITS.aspectWeight*Math.abs(Math.log(aspect/p.aspect))+
src\lib\engine\planner\index.ts:96: PLAN_LEARNING_LIMITS.livingWeight*Math.abs(livingY-p.living[1])+
src\lib\engine\planner\index.ts:97: PLAN_LEARNING_LIMITS.kitchenWeight*Math.abs(kitchenY-p.kitchen[1])
src\lib\engine\planner\index.ts:120: if (roofReady.length === (request.proposal ? PLAN_LEARNING_LIMITS.candidatePool : 4)) break
src\lib\engine\planner\index.ts:123: const pool = eligible.filter((c) => c.score <= eligible[0].score + Math.abs(eligible[0].score) * 0.18).slice(0, request.proposal ? PLAN_LEARNING_LIMITS.candidatePool : 4)
